# The Mathematics of Gacha Systems
## Probability, pity mechanics, guarantees and expected costs in *Honkai: Star Rail*

**Research question:** How do the probability of obtaining a desired character and the expected cost change as the number of pulls increases, and how much do pity and guarantee mechanics affect these outcomes?

**Scope:** Character Event Warp. The notebook models 5-star characters and 4-star-or-higher guarantees, then compares theoretical calculations with Monte Carlo simulations.

> **Important modelling note:** Officially published base rates and hard-pity limits are distinguished from community-derived or simplified assumptions. The flat 32.4% soft-pity rate used below is an intentionally simplified adaptation of a user-selected Genshin Impact community model. It is not an official Honkai: Star Rail rate curve.


## 1. Data, rules and assumptions

### 1.1 Character Event Warp rules used in this project

| Mechanic | Model value | Status |
|---|---:|---|
| Base 5-star probability | 0.6% | Official base rate |
| Consolidated 5-star probability | 1.6% | Official published rate |
| 5-star hard pity | 90 pulls | Official guarantee |
| Soft pity begins | Pull 74 | Community-observed convention |
| Featured 5-star chance | 50% | Warp rule |
| Featured guarantee after losing 50/50 | Next 5-star is featured | Warp rule |
| Pull cost | 160 Stellar Jade | In-game conversion |
| 4-star-or-higher guarantee | At least one in 10 pulls | Warp rule |

### 1.2 Soft-pity assumption

For a transparent first model, we use:

- Pulls 1–73: 0.6% conditional chance of a 5-star.
- Pulls 74–89: 32.4% conditional chance of a 5-star.
- Pull 90: 100% chance of a 5-star.

This is a deliberately simplified approximation. It does **not** reproduce the official consolidated rate exactly and should not be presented as the exact in-game probability curve. A later sensitivity analysis can compare it with a gradually increasing soft-pity curve.

### 1.3 Four-star modelling assumption

The official guarantee is expressed as **at least one 4-star-or-higher result within 10 pulls**. A 5-star result can also satisfy that condition. To keep the first model tractable, we use a separate 4-star guarantee counter and assume:

- Base 4-star probability: 5.1% (combined character and Light Cone rate supplied for the project).
- If no 4-star-or-higher result has occurred in the previous nine pulls, the next pull is guaranteed to be 4-star-or-higher.
- On a 4-star result, the 4-star pity counter resets.
- A 5-star result also counts as 4-star-or-higher and resets the 4-star counter.

**Limitation:** The exact interaction between 4-star and 5-star guarantees, and the split between 4-star characters and Light Cones, should be checked against the official Warp details before treating this simplified 4-star model as an exact reproduction of the game.

### 1.4 Featured 4-star modelling

The initial model focuses on the probability of receiving **any 4-star-or-higher item**, not a specific featured 4-star. We will not invent a featured 4-star probability without a verified banner-specific rate. This keeps the main results defensible.


In [ ]:
# 2. Imports and project settings

import random
import statistics
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_SEED = 42
N_SIMULATIONS = 100_000

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

BASE_5_STAR = 0.006
SOFT_PITY_RATE = 0.324
HARD_PITY_5_STAR = 90
SOFT_PITY_START = 74

BASE_4_STAR = 0.051
HARD_PITY_4_STAR = 10

JADE_PER_PULL = 160
USD_PER_PULL_ESTIMATE = None  # Fill in after selecting the package-pricing method.

print("Simulation count:", N_SIMULATIONS)
print("Random seed:", RANDOM_SEED)


## 3. Theoretical probability model

Let \(p_i\) be the conditional probability of receiving a 5-star on pull \(i\), assuming no 5-star has appeared earlier in the current pity cycle.

The probability of receiving no 5-star in the first \(n\) pulls is:

\[
P(\text{no 5-star in }n)=\prod_{i=1}^{n}(1-p_i)
\]

Therefore:

\[
P(\text{at least one 5-star in }n)
=1-\prod_{i=1}^{n}(1-p_i)
\]

The probability of a featured 5-star is more complicated because it depends on the 50/50 state. The simulation below tracks that guarantee directly.


In [ ]:
# 3.1 Conditional 5-star probability for each pull

def five_star_probability(pity_count):
    \"\"\"Return the conditional 5-star probability for the next pull.

    pity_count is the number of consecutive pulls since the previous 5-star.
    Thus pity_count=0 means the next pull is pull 1 of a new pity cycle.
    \"\"\"
    next_pull = pity_count + 1

    if next_pull >= HARD_PITY_5_STAR:
        return 1.0
    if next_pull >= SOFT_PITY_START:
        return SOFT_PITY_RATE
    return BASE_5_STAR


def probability_any_five_star(number_of_pulls):
    \"\"\"Theoretical chance of at least one 5-star within n pulls.\"\"\"
    survival_probability = 1.0
    pity_count = 0

    for _ in range(number_of_pulls):
        p = five_star_probability(pity_count)
        survival_probability *= (1 - p)
        pity_count += 1
        if p == 1.0:
            survival_probability = 0.0
            break

    return 1 - survival_probability


# Check selected pull counts
for pulls in [10, 20, 50, 72, 88, 90]:
    print(f"{pulls:>3} pulls: {probability_any_five_star(pulls):.2%}")


In [ ]:
# 3.2 Plot probability of at least one 5-star

pull_counts = range(1, HARD_PITY_5_STAR + 1)
five_star_chances = [probability_any_five_star(n) for n in pull_counts]

plt.figure(figsize=(10, 5))
plt.plot(pull_counts, five_star_chances, label="With simplified pity")
plt.axvline(73, linestyle="--", label="Soft pity starts")
plt.axvline(90, linestyle=":", label="Hard pity")
plt.xlabel("Number of pulls")
plt.ylabel("Probability of at least one 5-star")
plt.title("Chance of obtaining at least one 5-star")
plt.ylim(0, 1.02)
plt.grid(True)
plt.legend()
plt.show()


## 4. Featured 5-star character: 50/50 and guarantee

When a 5-star is obtained:

1. If the featured guarantee is active, the character is featured.
2. Otherwise, the result is a 50/50: featured or non-featured.
3. If the 50/50 is lost, the next 5-star is guaranteed to be featured.
4. The 5-star pity counter resets after every 5-star.

The simulation tracks both pity and the featured guarantee for every simulated player.


In [ ]:
def simulate_character_warp(max_pulls, rng=None):
    \"\"\"Simulate one player's Character Event Warp pulls.

    Returns a dictionary describing 5-star and 4-star outcomes.
    The simplified 4-star model treats a 5-star as satisfying the
    4-star-or-higher guarantee.
    \"\"\"
    if rng is None:
        rng = random

    pity5 = 0
    pity4 = 0
    guaranteed_featured = False
    five_star_count = 0
    featured_count = 0
    four_star_or_higher_count = 0
    first_featured_pull = None
    first_five_star_pull = None

    for pull_number in range(1, max_pulls + 1):
        p5 = five_star_probability(pity5)

        # 5-star check
        if rng.random() < p5:
            five_star_count += 1
            four_star_or_higher_count += 1
            if first_five_star_pull is None:
                first_five_star_pull = pull_number

            if guaranteed_featured:
                is_featured = True
                guaranteed_featured = False
            else:
                is_featured = (rng.random() < 0.50)
                if not is_featured:
                    guaranteed_featured = True

            if is_featured:
                featured_count += 1
                if first_featured_pull is None:
                    first_featured_pull = pull_number

            pity5 = 0
            pity4 = 0
            continue

        # No 5-star: check 4-star-or-higher guarantee.
        pity4 += 1
        guaranteed_four_star = (pity4 >= HARD_PITY_4_STAR)

        if guaranteed_four_star or rng.random() < BASE_4_STAR:
            four_star_or_higher_count += 1
            pity4 = 0

        pity5 += 1

    return {
        "pulls": max_pulls,
        "five_star_count": five_star_count,
        "featured_count": featured_count,
        "four_star_or_higher_count": four_star_or_higher_count,
        "first_five_star_pull": first_five_star_pull,
        "first_featured_pull": first_featured_pull,
        "featured_guarantee_active": guaranteed_featured,
    }


# Quick example
simulate_character_warp(72)


## 5. Monte Carlo simulation

We simulate 100,000 independent players for each saving scenario:

- Free-to-play: 72 Character Event Warp pulls.
- Monthly Pass: 88 pulls.
- Monthly Pass + Battle Pass: 97 pulls.

These are project-specific reward totals supplied for Version 4.5. They are not guaranteed monthly income values and may change between versions. Standard Passes are excluded because they cannot be used on the limited Character Event Warp.


In [ ]:
# 5.1 Run simulations

SCENARIOS = {
    "F2P": 72,
    "Monthly Pass": 88,
    "Monthly + BP": 97,
}

def run_many_simulations(number_of_players, pulls, seed=RANDOM_SEED):
    rng = random.Random(seed)
    results = [
        simulate_character_warp(pulls, rng)
        for _ in range(number_of_players)
    ]
    return pd.DataFrame(results)


simulation_results = {}
summary_rows = []

for index, (scenario, pulls) in enumerate(SCENARIOS.items()):
    results = run_many_simulations(
        N_SIMULATIONS,
        pulls,
        seed=RANDOM_SEED + index
    )
    simulation_results[scenario] = results

    summary_rows.append({
        "Scenario": scenario,
        "Pulls": pulls,
        "Chance of any 5-star": (results["five_star_count"] > 0).mean(),
        "Chance of featured 5-star": (results["featured_count"] > 0).mean(),
        "Average 5-stars": results["five_star_count"].mean(),
        "Average 4-star-or-higher results": results["four_star_or_higher_count"].mean(),
    })

summary = pd.DataFrame(summary_rows)
summary


In [ ]:
# 5.2 Compare theoretical and simulated chance of any 5-star

comparison_rows = []

for scenario, pulls in SCENARIOS.items():
    simulated_probability = (
        simulation_results[scenario]["five_star_count"] > 0
    ).mean()

    comparison_rows.append({
        "Scenario": scenario,
        "Pulls": pulls,
        "Theoretical probability": probability_any_five_star(pulls),
        "Simulated probability": simulated_probability,
        "Absolute difference": abs(
            probability_any_five_star(pulls) - simulated_probability
        ),
    })

comparison = pd.DataFrame(comparison_rows)
comparison


## 6. Four-star-or-higher guarantee

The simplified model uses a 5.1% base chance and forces a 4-star-or-higher result if nine consecutive pulls have passed without one. Because a 5-star also satisfies the guarantee, this section reports the combined outcome.

This is not the probability of receiving a specific 4-star character. A banner-specific featured 4-star model would require verified rate and guarantee details.


In [ ]:
# 6.1 Estimate the chance of at least one 4-star-or-higher result

four_star_rows = []

for scenario, results in simulation_results.items():
    pulls = SCENARIOS[scenario]
    chance = (results["four_star_or_higher_count"] > 0).mean()
    four_star_rows.append({
        "Scenario": scenario,
        "Pulls": pulls,
        "Chance of at least one 4-star-or-higher": chance,
        "Average 4-star-or-higher results": results[
            "four_star_or_higher_count"
        ].mean(),
    })

four_star_summary = pd.DataFrame(four_star_rows)
four_star_summary


## 7. Pull costs and worst-case guarantee

Each pull costs 160 Stellar Jade.

- One 5-star is guaranteed within 90 pulls: 14,400 Stellar Jade.
- A featured 5-star is guaranteed within 180 pulls if the first 5-star loses the 50/50: 28,800 Stellar Jade.

The USD cost depends on which top-up packages are purchased, whether first-time bonuses are available, and the player's existing Jade balance. We therefore keep USD pricing separate from the probability model until the package-purchase strategy is defined.


In [ ]:
# 7.1 Pull and Jade requirements

cost_table = pd.DataFrame([
    {"Goal": "One 5-star (hard pity)", "Pulls": 90},
    {"Goal": "Featured 5-star (worst case)", "Pulls": 180},
    {"Goal": "F2P Version 4.5 scenario", "Pulls": 72},
    {"Goal": "Monthly Pass Version 4.5 scenario", "Pulls": 88},
    {"Goal": "Monthly + BP Version 4.5 scenario", "Pulls": 97},
])

cost_table["Stellar Jade required"] = cost_table["Pulls"] * JADE_PER_PULL
cost_table


## 8. USD top-up package analysis

The following package values were transcribed from the user's in-game shop screenshot. They are treated as project input, not independently verified store prices. The calculation assumes 1 Traumsplitter converts to 1 Stellar Jade and 160 Jade buys one pull.

| Price (USD) | Base currency | Bonus | Total currency |
|---:|---:|---:|---:|
| $0.99 | 60 | 0 | 60 |
| $4.99 | 300 | 30 | 330 |
| $14.99 | 980 | 110 | 1,090 |
| $29.99 | 1,980 | 260 | 2,240 |
| $49.99 | 3,280 | 600 | 3,880 |
| $99.99 | 6,480 | 1,600 | 8,080 |

The next table calculates the effective cost per pull if the entire package value is converted into pulls. Actual purchases are indivisible, so this is a comparison of package efficiency, not the exact amount a player must spend for a particular target.


In [ ]:
# 8.1 Package efficiency (USD)

packages = pd.DataFrame([
    {"Price_USD": 0.99, "Base_currency": 60, "Bonus": 0},
    {"Price_USD": 4.99, "Base_currency": 300, "Bonus": 30},
    {"Price_USD": 14.99, "Base_currency": 980, "Bonus": 110},
    {"Price_USD": 29.99, "Base_currency": 1980, "Bonus": 260},
    {"Price_USD": 49.99, "Base_currency": 3280, "Bonus": 600},
    {"Price_USD": 99.99, "Base_currency": 6480, "Bonus": 1600},
])

packages["Total_currency"] = packages["Base_currency"] + packages["Bonus"]
packages["Pull_equivalent"] = packages["Total_currency"] / JADE_PER_PULL
packages["USD_per_pull"] = (
    packages["Price_USD"] / packages["Pull_equivalent"]
)

packages[["Price_USD", "Total_currency", "Pull_equivalent", "USD_per_pull"]]


**Interpretation:** The largest package has the lowest effective USD cost per pull in this screenshot, but it also requires the largest upfront payment. The model does not assume that every player should buy the largest package. A proper exact-spending model must account for current Jade balance, one-time bonus eligibility, repeat-purchase rules and package combinations.


## 9. Limitations

1. The soft-pity curve is a simplified community-derived approximation, not an official published conditional-rate table.
2. The 4-star model is simplified and should be checked against the exact in-game rules.
3. The model does not yet distinguish 4-star characters from 4-star Light Cones.
4. It does not model a particular banner's featured 4-star lineup.
5. The Version 4.5 reward scenarios are historical inputs, not a promise of recurring income.
6. USD costs are not yet calculated because package purchases are discrete and may include first-time bonuses.
7. The model assumes pulls are independent except for the pity and guarantee states explicitly tracked.


## 10. Conclusion

Complete this section after running the notebook. Discuss:

- How much pity changes the chance of receiving a 5-star compared with a constant 0.6% model.
- How the 50/50 and featured guarantee change the probability of obtaining the desired character.
- How the three saving scenarios compare.
- How closely the Monte Carlo results match the theoretical calculations.
- Why the soft-pity and 4-star assumptions limit how precisely the model can reproduce the actual game.

## 11. Sources

Add direct links and access dates for the official HoYoverse Warp details, the community source used for the soft-pity approximation, and the Version 4.5 reward data. Clearly label official information, community-derived estimates, user-collected data, and modelling assumptions.
